In [ ]:
import astropy.units as u
import matplotlib.colors as mcolors
from matplotlib.colors import Colormap, LinearSegmentedColormap, Normalize, PowerNorm
from skimage.color import lab2rgb

class PUNCHNorm(mcolors.Normalize):
    def __init__(self, gamma, vmin=None, vmax=None):
        super().__init__(vmin=vmin, vmax=vmax)
        self.gamma = gamma

    def __call__(self, value, clip=None):
        x, is_scalar = self.process_value(value)
        self.autoscale_None(x)
        
        abs_max = max(abs(self.vmin), abs(self.vmax))
        
        result = np.sign(x) * (np.abs(x) / abs_max) ** self.gamma
        result = (result + 1) / 2
        
        return np.ma.array(result, mask=np.isnan(x))

    def inverse(self, value):
        abs_max = max(abs(self.vmin), abs(self.vmax))
        y = value * 2 - 1  # back to [-1, 1]
        return np.sign(y) * (np.abs(y) ** (1 / self.gamma)) * abs_max

def _cmap_punch_diverging() -> LinearSegmentedColormap:
    black_lab  = np.array([0,    0,    0])
    orange_lab = np.array([50,   15,   50])
    white_lab  = np.array([100,  0,    0])
    blue_lab   = np.array([50,  -15,  -50])

    n = 256
    half = n // 2
    lab_colors = np.zeros((n, 3))

    for i in range(half):
        t = i / (half - 1)
        lab_colors[i] = blue_lab * (1 - t) + black_lab * t

    quarter = n // 4
    for i in range(half, half + quarter):
        t = (i - half) / (quarter - 1)
        lab_colors[i] = black_lab * (1 - t) + orange_lab * t

    for i in range(half + quarter, n):
        t = (i - half - quarter) / (quarter - 1)
        lab_colors[i] = orange_lab * (1 - t) + white_lab * t

    rgb_colors = lab2rgb(lab_colors.reshape(1, -1, 3)).reshape(n, 3)
    rgb_colors = np.clip(rgb_colors, 0, 1)
    return LinearSegmentedColormap.from_list("PUNCH_diverging", rgb_colors, N=n)

cmap_punch_diverging = _cmap_punch_diverging()

In [ ]:
fig, ax = plt.subplots(figsize=[13,10])
im = ax.imshow(cube_sub.data[0,:,:], norm=PUNCHNorm(1/2.2, vmin=-1e-13, vmax=1e-13), cmap=cmap_punch_diverging)
fig.colorbar(im)